In [5]:
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
import joblib

In [3]:
data = pd.read_csv("../dataset/nba_elo.csv", dtype={'teamId' : str})
data

,date,season,neutral,playoff,team1,team2,elo1_pre,elo2_pre,elo_prob1,elo_prob2,elo1_post,elo2_post,score1,score2,is_home
0,1946-11-01,1947,0,NaN,TRH,NYK,1300.000000,1300.000000,0.640065,0.359935,1293.276700,1306.723300,66,68,1
1,1946-11-01,1947,0,NaN,NYK,TRH,1300.000000,1300.000000,0.359935,0.640065,1306.723300,1293.276700,68,66,0
2,1946-11-02,1947,0,NaN,PRO,BOS,1300.000000,1300.000000,0.640065,0.359935,1305.154200,1294.845800,59,53,1
3,1946-11-02,1947,0,NaN,STB,PIT,1300.000000,1300.000000,0.640065,0.359935,1304.690800,1295.309200,56,51,1
4,1946-11-02,1947,0,NaN,CHS,NYK,1300.000000,1306.723300,0.631101,0.368899,1309.652100,1297.071200,63,47,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148933,2024-03-30,2024,0,NaN,MEM,ORL,1325.023208,1523.156879,0.152362,0.847638,1319.643511,1528.536576,88,118,0
148934,2024-03-30,2024,0,NaN,MIL,ATL,1586.179279,1493.120229,0.490012,0.509988,1596.162979,1483.136529,122,113,0
148935,2024-03-30,2024,0,NaN,NOP,BOS,1635.324867,1758.207851,0.467116,0.532884,1624.649366,1768.883352,92,104,1
148936,2024-03-30,2024,0,NaN,ORL,MEM,1523.156879,1325.023208,0.847638,0.152362,1528.536576,1319.643511,118,88,1


In [8]:
last_date = data['date'].iloc[-1]
elo_home = data[data['date1'] == "ATL"]
elo_away = data[data['date2'] == "ATL"]
last_date = datetime.strptime(data['date'].iloc[-1], '%Y-%m-%d') > datetime.strptime("2020-02-24", '%Y-%m-%d')
last_date

True

In [10]:
df_elo_home = data[data['team1'] == "ATL"]
df_elo_home

,date,season,neutral,playoff,team1,team2,elo1_pre,elo2_pre,elo_prob1,elo_prob2,elo1_post,elo2_post,score1,score2,is_home
17150,1968-10-16,1969,0,NaN,ATL,CIN,1521.127000,1495.579600,0.673203,0.326797,1500.975700,1515.730800,110,125,1
17168,1968-10-19,1969,0,NaN,ATL,MIL,1500.975700,1292.690700,0.855033,0.144967,1504.517900,1289.148400,125,107,1
17194,1968-10-23,1969,0,NaN,ATL,CHI,1504.517900,1389.985400,0.774683,0.225317,1509.696300,1384.807000,106,91,1
17216,1968-10-25,1969,0,NaN,ATL,SEA,1509.696300,1333.781600,0.607544,0.392456,1495.451400,1348.026500,112,123,0
17235,1968-10-27,1969,0,NaN,ATL,PHO,1495.451400,1315.742900,0.612739,0.387261,1508.606800,1302.587500,123,100,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148832,2024-03-23,2024,0,NaN,ATL,CHO,1465.499224,1303.486236,0.818804,0.181196,1473.745028,1295.240432,132,91,1
148862,2024-03-25,2024,0,NaN,ATL,BOS,1473.745028,1773.904953,0.240085,0.759915,1482.488755,1765.161227,120,118,1
148895,2024-03-27,2024,0,NaN,ATL,POR,1482.488755,1311.849757,0.826055,0.173945,1486.166853,1308.171659,120,106,1
148906,2024-03-28,2024,0,NaN,ATL,BOS,1486.166853,1765.161227,0.263012,0.736988,1493.120229,1758.207851,123,122,1


In [13]:
df_elo_away = data[data['team2'] == "ATL"]
df_elo_away

,date,season,neutral,playoff,team1,team2,elo1_pre,elo2_pre,elo_prob1,elo_prob2,elo1_post,elo2_post,score1,score2,is_home
17153,1968-10-16,1969,0,NaN,CIN,ATL,1495.579600,1521.127000,0.326797,0.673203,1515.730800,1500.975700,125,110,0
17175,1968-10-19,1969,0,NaN,MIL,ATL,1292.690700,1500.975700,0.144967,0.855033,1289.148400,1504.517900,107,125,0
17200,1968-10-23,1969,0,NaN,CHI,ATL,1389.985400,1504.517900,0.225317,0.774683,1384.807000,1509.696300,91,106,0
17212,1968-10-25,1969,0,NaN,SEA,ATL,1333.781600,1509.696300,0.392456,0.607544,1348.026500,1495.451400,123,112,1
17231,1968-10-27,1969,0,NaN,PHO,ATL,1315.742900,1495.451400,0.387261,0.612739,1302.587500,1508.606800,100,123,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148824,2024-03-23,2024,0,NaN,CHO,ATL,1303.486236,1465.499224,0.181196,0.818804,1295.240432,1473.745028,91,132,0
148851,2024-03-25,2024,0,NaN,BOS,ATL,1773.904953,1473.745028,0.759915,0.240085,1765.161227,1482.488755,118,120,0
148883,2024-03-27,2024,0,NaN,POR,ATL,1311.849757,1482.488755,0.173945,0.826055,1308.171659,1486.166853,106,120,0
148904,2024-03-28,2024,0,NaN,BOS,ATL,1765.161227,1486.166853,0.736988,0.263012,1758.207851,1493.120229,122,123,0


In [ ]:
if datetime.strptime(df_elo_home['date'].iloc[-1], '%Y-%m-%d') > datetime.strptime(df_elo_away['date'].iloc[-1], '%Y-%m-%d'):
    return df_elo_home['elo_post_1'].iloc[-1]
else :
    return df_elo_away['elo_post_2'].iloc[-1]